In [19]:
!pip install torch-geometric -q
!pip install ogb -q

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv

from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score

import numpy as np

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cuda


In [22]:
dataset = Planetoid(root='data/Pubmed', name='Pubmed')

data = dataset[0]

In [23]:
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True
)

train_data, val_data, test_data = transform(data)

print(train_data)

Data(x=[19717, 500], edge_index=[2, 75352], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717], edge_label=[75352], edge_label_index=[2, 75352])


In [24]:
from torch_geometric.utils import to_dense_adj

adj = to_dense_adj(train_data.edge_index)[0]

A2 = torch.mm(adj, adj)
A3 = torch.mm(A2, adj)

A2 = (A2 > 0).float()
A3 = (A3 > 0).float()

print(A2.shape)

torch.Size([19717, 19717])


In [25]:
class TeacherGNN(nn.Module):

    def __init__(self, in_channels, hidden=256):

        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, hidden)

    def encode(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(x, p=0.3, training=self.training)

        x = self.conv2(x, edge_index)

        return x

    def decode(self, z, edge_label_index):

        src, dst = edge_label_index

        return (z[src] * z[dst]).sum(dim=-1)

    def forward(self, x, edge_index, edge_label_index):

        z = self.encode(x, edge_index)

        return self.decode(z, edge_label_index)

In [26]:
class StudentMLP(nn.Module):

    def __init__(self, in_channels, hidden=256):

        super().__init__()

        self.lin1 = nn.Linear(in_channels, hidden)
        self.lin2 = nn.Linear(hidden, hidden)

    def encode(self, x):

        x = self.lin1(x)

        x = F.relu(x)

        x = F.dropout(x, p=0.3, training=self.training)

        x = self.lin2(x)

        return x

    def decode(self, z, edge_label_index):

        src, dst = edge_label_index

        return (z[src] * z[dst]).sum(dim=-1)

    def forward(self, x, edge_label_index):

        z = self.encode(x)

        return self.decode(z, edge_label_index)

In [27]:
class SAL(nn.Module):

    def __init__(self, num_nodes, hidden=256):

        super().__init__()

        self.fc1 = nn.Linear(num_nodes, hidden)

        self.fc2 = nn.Linear(hidden, hidden)

    def forward(self, A2, A3):

        s2 = F.relu(self.fc1(A2))

        s3 = F.relu(self.fc1(A3))

        s = 0.6 * s2 + 0.4 * s3

        s = self.fc2(s)

        return s

In [28]:
teacher = TeacherGNN(dataset.num_features).to(device)

student = StudentMLP(dataset.num_features).to(device)

sal = SAL(train_data.num_nodes).to(device)

In [29]:
optimizer_teacher = optim.Adam(
    teacher.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

optimizer_student = optim.Adam(
    list(student.parameters()) + list(sal.parameters()),
    lr=0.001,
    weight_decay=1e-4
)

In [30]:
def train_teacher():

    teacher.train()

    optimizer_teacher.zero_grad()

    z = teacher.encode(
        train_data.x.to(device),
        train_data.edge_index.to(device)
    )

    pos_edge = train_data.edge_label_index.to(device)

    neg_edge = negative_sampling(
        edge_index=train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=pos_edge.size(1)
    ).to(device)

    edge_label_index = torch.cat([pos_edge, neg_edge], dim=1)

    edge_label = torch.cat([
        torch.ones(pos_edge.size(1)),
        torch.zeros(neg_edge.size(1))
    ]).to(device)

    out = teacher.decode(z, edge_label_index)

    loss = F.binary_cross_entropy_with_logits(out, edge_label)

    loss.backward()

    optimizer_teacher.step()

    return loss.item()

In [31]:
for epoch in range(1, 501):

    loss = train_teacher()

    if epoch % 20 == 0:

        print(f"Teacher Epoch {epoch} Loss {loss:.4f}")

Teacher Epoch 20 Loss 0.6652
Teacher Epoch 40 Loss 0.6514
Teacher Epoch 60 Loss 0.6368
Teacher Epoch 80 Loss 0.6268
Teacher Epoch 100 Loss 0.6174
Teacher Epoch 120 Loss 0.6097
Teacher Epoch 140 Loss 0.6059
Teacher Epoch 160 Loss 0.6024
Teacher Epoch 180 Loss 0.5999
Teacher Epoch 200 Loss 0.5971
Teacher Epoch 220 Loss 0.5963
Teacher Epoch 240 Loss 0.5914
Teacher Epoch 260 Loss 0.5917
Teacher Epoch 280 Loss 0.5890
Teacher Epoch 300 Loss 0.5886
Teacher Epoch 320 Loss 0.5880
Teacher Epoch 340 Loss 0.5878
Teacher Epoch 360 Loss 0.5866
Teacher Epoch 380 Loss 0.5866
Teacher Epoch 400 Loss 0.5875
Teacher Epoch 420 Loss 0.5866
Teacher Epoch 440 Loss 0.5856
Teacher Epoch 460 Loss 0.5860
Teacher Epoch 480 Loss 0.5850
Teacher Epoch 500 Loss 0.5847


In [32]:
A2_device = A2.to(device)
A3_device = A3.to(device)

temperature = 2.0

alpha = 0.7
beta = 0.3
gamma = 0.2

def train_student():

    student.train()
    sal.train()

    optimizer_student.zero_grad()

    with torch.no_grad():

        teacher_z = teacher.encode(
            train_data.x.to(device),
            train_data.edge_index.to(device)
        )

    student_z = student.encode(
        train_data.x.to(device)
    )

    structural_attr = sal(A2_device, A3_device)

    enhanced_z = (
        0.7 * student_z +
        0.3 * structural_attr
    )

    pos_edge = train_data.edge_label_index.to(device)

    neg_edge = negative_sampling(
        edge_index=train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=pos_edge.size(1) * 2
    ).to(device)

    edge_label_index = torch.cat(
        [pos_edge, neg_edge],
        dim=1
    )

    edge_label = torch.cat([
        torch.ones(pos_edge.size(1)),
        torch.zeros(neg_edge.size(1))
    ]).to(device)

    teacher_out = teacher.decode(
        teacher_z,
        edge_label_index
    )

    student_out = student.decode(
        enhanced_z,
        edge_label_index
    )

    sup_loss = F.binary_cross_entropy_with_logits(
        student_out,
        edge_label
    )

    kd_loss = F.mse_loss(
        torch.sigmoid(student_out / temperature),
        torch.sigmoid(teacher_out / temperature)
    )

    structure_loss = F.mse_loss(
        enhanced_z,
        teacher_z
    )

    loss = (
        alpha * sup_loss +
        beta * kd_loss +
        gamma * structure_loss
    )

    loss.backward()

    optimizer_student.step()

    return loss.item()

In [33]:
best_loss = 999

for epoch in range(1, 601):

    loss = train_student()

    if epoch % 20 == 0:

        print(f"Student Epoch {epoch} Loss {loss:.4f}")

Student Epoch 20 Loss 0.4470
Student Epoch 40 Loss 0.4377
Student Epoch 60 Loss 0.4322
Student Epoch 80 Loss 0.4290
Student Epoch 100 Loss 0.4276
Student Epoch 120 Loss 0.4265
Student Epoch 140 Loss 0.4251
Student Epoch 160 Loss 0.4252
Student Epoch 180 Loss 0.4240
Student Epoch 200 Loss 0.4228
Student Epoch 220 Loss 0.4220
Student Epoch 240 Loss 0.4220
Student Epoch 260 Loss 0.4211
Student Epoch 280 Loss 0.4206
Student Epoch 300 Loss 0.4205
Student Epoch 320 Loss 0.4208
Student Epoch 340 Loss 0.4202
Student Epoch 360 Loss 0.4199
Student Epoch 380 Loss 0.4193
Student Epoch 400 Loss 0.4193
Student Epoch 420 Loss 0.4185
Student Epoch 440 Loss 0.4187
Student Epoch 460 Loss 0.4183
Student Epoch 480 Loss 0.4184
Student Epoch 500 Loss 0.4187
Student Epoch 520 Loss 0.4172
Student Epoch 540 Loss 0.4175
Student Epoch 560 Loss 0.4174
Student Epoch 580 Loss 0.4169
Student Epoch 600 Loss 0.4167


In [34]:
def hits_at_k(pos_pred, neg_pred, k):

    pos_pred = pos_pred.cpu()
    neg_pred = neg_pred.cpu()

    threshold = torch.topk(
        neg_pred,
        k
    ).values[-1]

    hits = (pos_pred > threshold).float().mean()

    return hits.item()

In [35]:
@torch.no_grad()

def evaluate(data_split):

    student.eval()
    sal.eval()

    z = student.encode(
        data_split.x.to(device)
    )

    structural_attr = sal(
        A2_device,
        A3_device
    )

    z = 0.7 * z + 0.3 * structural_attr

    edge_index = data_split.edge_label_index.to(device)

    pred = student.decode(z, edge_index)

    pred = torch.sigmoid(pred)

    edge_label = data_split.edge_label.to(device)

    auc = roc_auc_score(
        edge_label.cpu(),
        pred.cpu()
    )

    ap = average_precision_score(
        edge_label.cpu(),
        pred.cpu()
    )

    pos_pred = pred[edge_label == 1]

    neg_pred = pred[edge_label == 0]

    hit20 = hits_at_k(
        pos_pred,
        neg_pred,
        20
    )

    hit50 = hits_at_k(
        pos_pred,
        neg_pred,
        50
    )

    return auc, ap, hit20, hit50

In [36]:
auc, ap, hit20, hit50 = evaluate(test_data)

print("========== FINAL RESULT ==========")

print(f"AUC      : {auc:.4f}")
print(f"AP       : {ap:.4f}")
print(f"Hits@20  : {hit20:.4f}")
print(f"Hits@50  : {hit50:.4f}")

========== FINAL RESULT ==========
AUC      : 0.8358
AP       : 0.8773
Hits@20  : 0.4522
Hits@50  : 0.5593
